### 0. Load packages

In [1]:
# load default packages
import numpy as np
import pandas as pd
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# load Naive Bayes model & helper functions
from utils import *
from naive_bayes import MultinomialNaiveBayes, GaussianNaiveBayes

### 1. Weather dataset

In [2]:
# test with small data
df = pd.read_csv("data/tennis.txt")
df = df.drop(['Day'], axis = 1)


In [3]:
cls = MultinomialNaiveBayes()
cls.fit(df[['Outlook','Humidity','Wind']], df['Play'])
preds = cls.predict(df[['Outlook','Humidity','Wind']])
accuracy(preds, df['Play'])

0.9285714285714286

### 2. Spam email dataset

Detecting spam email with Multinomial Naive Bayes

In [4]:
df = pd.read_csv("data/messages.csv") # source: kaggle.com/datasets/mandygu/lingspam-dataset/
df

,subject,message,label
0,job posting - apple-iss research center,content - length : 3386 apple-iss research cen...,0
1,NaN,"lang classification grimes , joseph e . and ba...",0
2,query : letter frequencies for text identifica...,i am posting this inquiry for sergei atamas ( ...,0
3,risk,a colleague and i are researching the differin...,0
4,request book information,earlier this morning i was on the phone with a...,0
...,...,...,...
2888,love your profile - ysuolvpv,hello thanks for stopping by ! ! we have taken...,1
2889,you have been asked to join kiddin,"the list owner of : "" kiddin "" has invited you...",1
2890,anglicization of composers ' names,"judging from the return post , i must have sou...",0
2891,"re : 6 . 797 , comparative method : n - ary co...",gotcha ! there are two separate fallacies in t...,0


- Preprocess text: removing numbers, symbols, etc., removing stopwords, stemming

- Convert text into term frequency matrix (Bag-of-Words)

In [5]:
def preprocess_text(text):
    
    text = text.lower()

    tokens = re.findall(r'\b[a-z]{2,}\b', text)

    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]

    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]

    return tokens


# replicate sklearn CountVectorizer 

def count_vectorizer(corpus):

    tokenized = []
    vocab_set = set() # collect all unique words
    
    for doc in corpus:
        tokens = preprocess_text(doc)
        tokenized.append(tokens)
        vocab_set.update(tokens)

    vocab = sorted(vocab_set)
    
    # create document-term matrix
    matrix = []
    for tokens in tokenized:
        word_counts = Counter(tokens)
        row = [word_counts.get(word, 0) for word in vocab]
        matrix.append(row)

    return pd.DataFrame(matrix, columns=vocab)


In [6]:
df_test, df_train = train_test_split(df, 0.2, 'label')

In [7]:
# preprocess text
X_train = count_vectorizer(df_train['message'])  
y_train = pd.Series(df_train['label'], name="target_label")

X_test = count_vectorizer(df_test['message'])  
y_test = pd.Series(df_test['label'], name="target_label")

In [8]:
X_train

,aa,aaa,aaai,aaal,aaarghh,aabb,aabyhoej,aac,aachen,aafarli,...,zweitsprach,zwicki,zwiep,zwischen,zwitserlood,zybatov,zybatow,zygmunt,zytkow,zzlsa
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2309,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2310,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2311,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2312,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
nb = MultinomialNaiveBayes()
nb.fit(X_train, y_train)

In [12]:
# Confusion matrix
pd.crosstab(preds, pd.Series(y_test), rownames=['Actual'], colnames=['Predicted'], margins=True)

Predicted,0,1,All
Actual,,,
0,487,15,502
1,0,77,77
All,487,92,579


### 3. Iris dataset
Deal with continuous features

In [13]:
from sklearn.datasets import load_iris
import pandas as pd

iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='target')

In [16]:
cls = GaussianNaiveBayes()
cls.fit(X, y)
preds = cls.predict(X)
accuracy(preds, y)

0.96